### Đọc dữ liệu

In [15]:
import pandas as pd

df_gmv = pd.read_csv("./cleaned_data/gmv_clean_sales_infor.csv", encoding="utf-8-sig")
df_leads = pd.read_csv("./cleaned_data/leads_clean_with_note_phone.csv", encoding="utf-8-sig")

### Chuẩn hóa cột Phone_clean trong cả 2 bảng

In [16]:
import pandas as pd

def normalize_phone(x):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x

df_gmv["Phone_clean"] = df_gmv["Phone_clean"].apply(normalize_phone)
df_leads["Phone_clean"] = df_leads["Phone_clean"].apply(normalize_phone)

### Tìm bản match bằng Phone_clean trước, sau đó ưu tiên bởi Sales_infor trong gmv và TVTS trong leads

In [17]:
import numpy as np

# ====================================================
# Chuẩn hóa dữ liệu
# ====================================================

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

for col in ["Phone_clean", "Sales_infor"]:
    df_gmv[col] = (
        df_gmv[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

for col in ["Phone_clean", "TVTS"]:
    df_leads[col] = (
        df_leads[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# ID để nhận diện từng record GMV
df_gmv["_gmv_id"] = np.arange(len(df_gmv))

# ====================================================
# Join theo Phone_clean
# ====================================================

candidate = df_gmv.merge(
    df_leads,
    on="Phone_clean",
    how="left",
    suffixes=("_gmv", "_lead")
)

# ====================================================
# Ưu tiên Sales_infor == TVTS
# ====================================================

candidate["_priority"] = (
    candidate["Sales_infor"] == candidate["TVTS"]
).astype(int)

# Sắp xếp:
# 1. cùng Sales
# 2. giữ nguyên thứ tự xuất hiện trong leads
candidate = candidate.sort_values(
    by=["_gmv_id", "_priority"],
    ascending=[True, False],
    kind="stable"
)

# ====================================================
# Mỗi GMV chỉ lấy đúng 1 Lead
# ====================================================

matched = (
    candidate
    .drop_duplicates("_gmv_id", keep="first")
    .drop(columns=["_priority"])
)

# ====================================================
# Thống kê
# ====================================================

matched_count = matched["Phone_clean"].ne("").sum() - matched["TVTS"].isna().sum()

print(f"GMV records: {len(df_gmv)}")
print(f"Matched records: {matched['TVTS'].notna().sum()}")
print(f"Unmatched records: {matched['TVTS'].isna().sum()}")

matched.head()

GMV records: 463
Matched records: 281
Unmatched records: 182


,bank day,bank time,Gateway,User Name,Phone,UID_gmv,Package,Fixed/ non-fixed,Pay Time,Real Pay(VND),...,Ngày lên học thử,Current Binded Sale,Sales in latest system-weighted allocation,Latest system-weighted Allocation Time,Allocate reason,Tên sale chatpage,KOL,Check Ad_ID,UID_clean_lead,Phone_extracted_from_note
0,2026/5/2,18:28,VP Bank,C Thanh - Bé Hân,84-908224672,3311001921,2/W- NEW 96 PHI+10,2/W,2026/5/2,18.100.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026/5/11,NaN,NaN,Chị Nga - Bé Khang,84-909517679,3179205818,2/W- NEW 24 PHI+2,2/W,2026/5/11,4.680.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026/5/11,20:33,BIDV,Chị Diễn - Bé Thư Di,84-964678701,3311304274,2/W- NEW 24 PHI+2,2/W,2026/5/11,4.680.000,...,2026-05-11,Lai Ngoc Thuy Linh,Lai Ngoc Thuy Linh,2026-05-09 22:52:34,Weighted,HCM - LinhLNT,NaN,NaN,3311304274,NaN
3,2026/5/14,09:30,Agribank,Chị Huệ - Bé Thi,84-389970625,3310902627,2/W- Both AB (A-PH) REFER 96 PHI+10,2/W,2026/5/14,17.490.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026/5/14,17:59,Techcombank,Chị Ngọc Anh - Bé Nhân,84-938593961,3309271098,2/W- NEW 24 PHI+2,2/W,2026/5/14,4.900.000,...,NaN,Lai Ngoc Thuy Linh,Lai Ngoc Thuy Linh,Lai Ngoc Thuy Linh,Weighted,HCM - LinhLNT,NaN,NaN,3309271098,NaN


In [18]:
# Các GMV đã match thành công
matched_gmv_ids = matched.loc[
    matched["TVTS"].notna(),
    "_gmv_id"
]

# Giữ lại các GMV chưa match
df_gmv_unmatched = df_gmv[
    ~df_gmv["_gmv_id"].isin(matched_gmv_ids)
].copy()

print(f"Matched: {len(matched_gmv_ids)}")
print(f"Remaining unmatched: {len(df_gmv_unmatched)}")

Matched: 281
Remaining unmatched: 182


In [19]:
df_gmv = df_gmv[
    ~df_gmv["_gmv_id"].isin(matched_gmv_ids)
].copy()

In [21]:
print(df_gmv.shape)

(182, 41)


### Sử dụng Phone_extracted_from_note để tiếp tục tìm bản match

In [22]:
df_leads["Phone_extracted_from_note"] = df_leads["Phone_extracted_from_note"].apply(normalize_phone)

In [23]:
df_leads["Phone_extracted_from_note"].head()

0              
1              
2    0965468525
3    0985951781
4              
Name: Phone_extracted_from_note, dtype: object

In [24]:
# ====================================================
# Chuẩn hóa dữ liệu
# ====================================================

df_gmv = df_gmv.copy()
df_leads = df_leads.copy()

for col in ["Phone_clean", "Sales_infor"]:
    df_gmv[col] = (
        df_gmv[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

for col in ["Phone_extracted_from_note", "TVTS"]:
    df_leads[col] = (
        df_leads[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# ID duy nhất cho từng record GMV
df_gmv["_gmv_id"] = np.arange(len(df_gmv))

# ====================================================
# Match Phone_clean (GMV) <-> Phone_extracted_from_note (Leads)
# ====================================================

candidate = df_gmv.merge(
    df_leads,
    left_on="Phone_clean",
    right_on="Phone_extracted_from_note",
    how="left",
    suffixes=("_gmv", "_lead")
)

# ====================================================
# Ưu tiên Sales_infor == TVTS
# ====================================================

candidate["_priority"] = (
    candidate["Sales_infor"] == candidate["TVTS"]
).astype(int)

candidate = candidate.sort_values(
    by=["_gmv_id", "_priority"],
    ascending=[True, False],
    kind="stable"
)

matched = (
    candidate
    .drop_duplicates("_gmv_id", keep="first")
    .drop(columns="_priority")
)

# ====================================================
# Thống kê
# ====================================================

matched_records = matched[matched["TVTS"].notna()]

print(f"Matched: {len(matched_records)}")
print(f"Unmatched: {len(matched) - len(matched_records)}")

Matched: 3
Unmatched: 179


In [25]:
matched_gmv_ids = matched.loc[
    matched["TVTS"].notna(),
    "_gmv_id"
]

df_gmv = df_gmv[
    ~df_gmv["_gmv_id"].isin(matched_gmv_ids)
].copy()

df_gmv.drop(columns="_gmv_id", inplace=True, errors="ignore")

In [26]:
print(df_gmv.shape)

(179, 40)


### Deeper mining